# Linear Algebra for Machine Learning

Linear algebra is the **language** of machine learning. Every dataset is a matrix, every data point is a vector,
and every model operation (prediction, transformation, dimensionality reduction) is a matrix operation.

This notebook builds your intuition from the ground up — with pictures for everything.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

plt.rcParams['figure.figsize'] = (6, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 12

---
## 1. Vectors — Arrows in Space

A **vector** is just an ordered list of numbers. But geometrically, think of it as an **arrow** pointing
from the origin to a point in space.

- A 2D vector `[3, 2]` means: go 3 units right, 2 units up.
- In ML, a vector can represent a single data point (e.g., `[height, weight]` of a person).

**Key operations:**
- **Addition**: combine two arrows tip-to-tail
- **Scalar multiplication**: stretch or shrink an arrow

In [ ]:
def plot_vector(ax, v, origin=(0, 0), color='blue', label=None):
    ax.annotate('', xy=(origin[0]+v[0], origin[1]+v[1]), xytext=origin,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    if label:
        ax.text(origin[0]+v[0]/2 + 0.15, origin[1]+v[1]/2 + 0.15,
                label, fontsize=12, color=color, fontweight='bold')

a = np.array([3, 2])
b = np.array([1, 3])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.set_title('Two Vectors')
plot_vector(ax, a, color='blue', label='a = [3,2]')
plot_vector(ax, b, color='red', label='b = [1,3]')
ax.set_xlim(-1, 5); ax.set_ylim(-1, 5); ax.set_aspect('equal')

ax = axes[1]
ax.set_title('Vector Addition: a + b')
plot_vector(ax, a, color='blue', label='a')
plot_vector(ax, b, origin=tuple(a), color='red', label='b')
plot_vector(ax, a + b, color='green', label='a+b')
ax.set_xlim(-1, 6); ax.set_ylim(-1, 6); ax.set_aspect('equal')

ax = axes[2]
ax.set_title('Scalar Multiplication: 2a')
plot_vector(ax, a, color='blue', label='a')
plot_vector(ax, 2*a, color='purple', label='2a')
plot_vector(ax, 0.5*a, color='orange', label='0.5a')
ax.set_xlim(-1, 7); ax.set_ylim(-1, 5); ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 2. Dot Product — Measuring Similarity

The **dot product** of two vectors tells you how much they point in the same direction.

$$\mathbf{a} \cdot \mathbf{b} = |\mathbf{a}| \, |\mathbf{b}| \, \cos(\theta)$$

- If the vectors point the **same way** → large positive dot product
- If they're **perpendicular** → dot product is 0
- If they point in **opposite directions** → large negative dot product

**In ML:** This is how models measure **similarity** between data points. Cosine similarity is just the normalized dot product.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pairs = [
    (np.array([3, 1]), np.array([2, 3]), 'Small angle\n(similar)'),
    (np.array([3, 0]), np.array([0, 3]), 'Right angle\n(perpendicular)'),
    (np.array([3, 1]), np.array([-2, -1]), 'Opposite\n(dissimilar)'),
]

for ax, (v1, v2, title) in zip(axes, pairs):
    plot_vector(ax, v1, color='blue', label='a')
    plot_vector(ax, v2, color='red', label='b')

    dot = np.dot(v1, v2)
    cos_angle = dot / (np.linalg.norm(v1) * np.linalg.norm(v2))
    angle_deg = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))

    ax.set_title(f'{title}\na·b = {dot}, θ = {angle_deg:.0f}°')
    ax.set_xlim(-4, 4); ax.set_ylim(-3, 4); ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print(f"Dot product with numpy: np.dot([3,1], [2,3]) = {np.dot([3,1], [2,3])}")

### Projection — The Geometric View of Dot Product

The dot product also gives you the **projection** of one vector onto another:
how much of vector **a** lies along the direction of **b**.

In [ ]:
a = np.array([4, 3])
b = np.array([5, 0])

proj_scalar = np.dot(a, b) / np.linalg.norm(b)
proj_vector = (np.dot(a, b) / np.dot(b, b)) * b

fig, ax = plt.subplots(figsize=(7, 5))
plot_vector(ax, a, color='blue', label='a')
plot_vector(ax, b, color='red', label='b')
plot_vector(ax, proj_vector, color='green', label='proj')

ax.plot([a[0], proj_vector[0]], [a[1], proj_vector[1]],
        'k--', lw=1, label='perpendicular drop')

ax.set_title(f'Projection of a onto b\nProjection length = {proj_scalar:.2f}')
ax.set_xlim(-1, 6); ax.set_ylim(-1, 5); ax.set_aspect('equal')
ax.legend()
plt.show()

---
## 3. Matrices — Transformations in Disguise

A **matrix** is a rectangular grid of numbers. But the best way to think about it:
a matrix is a **transformation machine**.

Feed a vector in → get a transformed vector out.

| Matrix | What it does |
|--------|-------------|
| `[[2,0],[0,2]]` | Scales everything by 2 |
| `[[0,-1],[1,0]]` | Rotates 90° counterclockwise |
| `[[1,0],[0,-1]]` | Reflects across x-axis |

**In ML:** Your dataset is a matrix (rows = data points, columns = features).
Weight matrices in neural networks transform inputs to outputs.

In [ ]:
A = np.array([[2, 0], [0, 2]])
print(f"Matrix A (scaling):\n{A}")
print(f"\nA shape: {A.shape}")
print(f"A transpose:\n{A.T}")

B = np.array([[1, 2], [3, 4]])
C = np.array([[5, 6], [7, 8]])
print(f"\nMatrix multiplication B @ C:\n{B @ C}")
print(f"Note: B @ C ≠ C @ B (matrix multiplication is NOT commutative)")
print(f"C @ B:\n{C @ B}")

---
## 4. Matrix-Vector Multiplication — Transforming Points

When you multiply a matrix by a vector, you **transform** that vector.
Let's see this visually: take a shape made of points, and watch what the matrix does to it.

In [ ]:
theta = np.linspace(0, 2*np.pi, 50)
house_x = np.array([0, 1, 1, 0.5, 0, 0])
house_y = np.array([0, 0, 1, 1.5, 1, 0])
points = np.vstack([house_x, house_y])

transforms = {
    'Scaling (2x, 1x)': np.array([[2, 0], [0, 1]]),
    'Rotation (45°)': np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                                [np.sin(np.pi/4),  np.cos(np.pi/4)]]),
    'Shear': np.array([[1, 0.5], [0, 1]]),
    'Reflection (y-axis)': np.array([[-1, 0], [0, 1]]),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, M) in zip(axes, transforms.items()):
    transformed = M @ points
    ax.plot(points[0], points[1], 'b-o', markersize=4, label='Original')
    ax.plot(transformed[0], transformed[1], 'r-o', markersize=4, label='Transformed')
    ax.set_title(name)
    ax.set_xlim(-3, 3); ax.set_ylim(-1, 3); ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)

plt.tight_layout()
plt.show()

---
## 5. Identity Matrix & Inverse — Doing Nothing and Undoing

The **identity matrix** is the "do nothing" transformation — multiply any vector by it and you get the same vector back.

$$I = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$

The **inverse** of a matrix $A^{-1}$ **undoes** the transformation of $A$:

$$A \cdot A^{-1} = I$$

Not all matrices have an inverse — if a matrix squashes space into a lower dimension (determinant = 0), you can't unsquash it.

In [ ]:
I = np.eye(2)
print(f"Identity matrix:\n{I}")

A = np.array([[2, 1], [1, 3]])
A_inv = np.linalg.inv(A)
print(f"\nA:\n{A}")
print(f"\nA inverse:\n{A_inv}")
print(f"\nA @ A_inv (should be identity):\n{np.round(A @ A_inv, 10)}")

v = np.array([3, 2])
transformed = A @ v
recovered = A_inv @ transformed

fig, ax = plt.subplots(figsize=(6, 5))
plot_vector(ax, v, color='blue', label='original v')
plot_vector(ax, transformed, color='red', label='A @ v')
plot_vector(ax, recovered, color='green', label='A⁻¹ @ (A @ v)')
ax.set_title('Inverse undoes the transformation')
ax.set_xlim(-1, 9); ax.set_ylim(-1, 10); ax.set_aspect('equal')
ax.legend()
plt.show()

---
## 6. Determinant — How Much Space Gets Scaled

The **determinant** of a matrix tells you how much the transformation **scales area**.

- $\det(A) = 2$ → the transformation doubles all areas
- $\det(A) = 0$ → the transformation squashes everything onto a line (or point) — you've lost information
- $\det(A) < 0$ → the transformation flips orientation (like a mirror)

Geometrically: apply the matrix to a unit square and measure the area of the parallelogram you get.

In [ ]:
from matplotlib.patches import Polygon

matrices = [
    (np.array([[2, 0], [0, 1.5]]), 'Stretch'),
    (np.array([[1, 1], [0, 1]]), 'Shear'),
    (np.array([[1, 2], [2, 4]]), 'Singular (det=0)'),
]

unit_square = np.array([[0, 0], [1, 0], [1, 1], [0, 1]])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (M, name) in zip(axes, matrices):
    det = np.linalg.det(M)
    transformed = (M @ unit_square.T).T

    ax.add_patch(Polygon(unit_square, alpha=0.3, color='blue', label='Unit square (area=1)'))
    ax.add_patch(Polygon(transformed, alpha=0.3, color='red', label=f'Transformed (area={abs(det):.1f})'))

    ax.set_title(f'{name}\ndet = {det:.1f}')
    ax.set_xlim(-1, 5); ax.set_ylim(-1, 5); ax.set_aspect('equal')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("If det = 0, the matrix has NO inverse (information is lost).")

---
## 7. Eigenvalues & Eigenvectors — The DNA of a Transformation

When you apply a matrix transformation, most vectors change direction. But some special vectors
only get **stretched or shrunk** — they stay on the same line. These are **eigenvectors**,
and the stretch factor is the **eigenvalue**.

$$A \mathbf{v} = \lambda \mathbf{v}$$

- $\mathbf{v}$ is the eigenvector (the direction that survives)
- $\lambda$ is the eigenvalue (how much it stretches)

**In ML:** PCA (Principal Component Analysis) finds the eigenvectors of the data's covariance matrix.
These are the **most important directions** in your data — the directions of maximum variance.

In [ ]:
A = np.array([[3, 1], [0, 2]])
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Matrix A:\n{A}")
print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvectors (columns):\n{eigenvectors}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['blue', 'red']
for i in range(2):
    ax = axes[i]
    ev = eigenvectors[:, i] * 2
    transformed = A @ ev

    plot_vector(ax, ev, color=colors[i], label=f'eigenvector v{i+1}')
    plot_vector(ax, transformed, color='green',
               label=f'A @ v{i+1} = {eigenvalues[i]:.1f} × v{i+1}')

    ax.set_title(f'Eigenvector {i+1}\nλ = {eigenvalues[i]:.1f} (scales by {eigenvalues[i]:.1f}x)')
    ax.set_xlim(-2, 8); ax.set_ylim(-3, 5); ax.set_aspect('equal')
    ax.legend()
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)

plt.suptitle('Eigenvectors stay on their line — they only get stretched!', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Eigenvectors in PCA (Preview)

Here's a sneak peek: given 2D data, PCA finds the direction of maximum spread (the first eigenvector
of the covariance matrix). This becomes the "most important" feature.

In [ ]:
np.random.seed(42)
mean = [0, 0]
cov = [[3, 2], [2, 2]]
data = np.random.multivariate_normal(mean, cov, 200)

cov_matrix = np.cov(data.T)
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(data[:, 0], data[:, 1], alpha=0.4, s=15)

for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    scaled = vec * np.sqrt(val) * 2
    ax.annotate('', xy=scaled, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color=['red', 'blue'][i], lw=3))
    ax.text(scaled[0]+0.2, scaled[1]+0.2,
            f'PC{i+1} (λ={val:.2f})', fontsize=11, color=['red', 'blue'][i], fontweight='bold')

ax.set_title('PCA: Eigenvectors show the directions of maximum variance')
ax.set_aspect('equal')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
plt.show()

---
## 8. Norms — Measuring the Size of Vectors

A **norm** measures the "length" or "size" of a vector. Different norms measure size differently.

| Norm | Formula | Intuition |
|------|---------|----------|
| **L1** (Manhattan) | $\sum |x_i|$ | Distance walking on a grid (taxi cab) |
| **L2** (Euclidean) | $\sqrt{\sum x_i^2}$ | Straight-line distance |

**In ML:** Regularization penalizes large weights to prevent overfitting:
- **L1 regularization** (Lasso): uses L1 norm → pushes weights to **exactly zero** (feature selection)
- **L2 regularization** (Ridge): uses L2 norm → pushes weights to be **small but not zero**

In [ ]:
v = np.array([3, 4])
l1 = np.linalg.norm(v, ord=1)
l2 = np.linalg.norm(v, ord=2)
print(f"Vector: {v}")
print(f"L1 norm (|3| + |4|) = {l1}")
print(f"L2 norm (√(9+16))  = {l2}")

In [ ]:
theta = np.linspace(0, 2 * np.pi, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

t = np.linspace(0, 1, 500)
l1_x = np.concatenate([t, 1-t, -t, t-1])
l1_y = np.concatenate([1-t, -t, t-1, t])
axes[0].plot(l1_x, l1_y, 'b-', lw=2)
axes[0].fill(l1_x, l1_y, alpha=0.2, color='blue')
axes[0].set_title('L1 Unit "Circle" (Diamond)\n|x| + |y| = 1')
axes[0].set_aspect('equal')
axes[0].set_xlim(-1.5, 1.5); axes[0].set_ylim(-1.5, 1.5)
axes[0].axhline(0, color='gray', lw=0.5); axes[0].axvline(0, color='gray', lw=0.5)

axes[1].plot(np.cos(theta), np.sin(theta), 'r-', lw=2)
axes[1].fill(np.cos(theta), np.sin(theta), alpha=0.2, color='red')
axes[1].set_title('L2 Unit Circle\nx² + y² = 1')
axes[1].set_aspect('equal')
axes[1].set_xlim(-1.5, 1.5); axes[1].set_ylim(-1.5, 1.5)
axes[1].axhline(0, color='gray', lw=0.5); axes[1].axvline(0, color='gray', lw=0.5)

plt.suptitle('The shape of the "unit circle" depends on which norm you use', fontsize=13)
plt.tight_layout()
plt.show()

### Why L1 Produces Sparsity

The L1 diamond has corners that sit on the axes. When you minimize a loss function
subject to an L1 constraint, the solution tends to land on a **corner** — where one
or more coordinates are exactly zero. That's why L1 regularization does feature selection!

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x_grid = np.linspace(-2, 4, 200)
y_grid = np.linspace(-2, 4, 200)
X, Y = np.meshgrid(x_grid, y_grid)
Z = (X - 2)**2 + (Y - 2)**2

for ax, (norm_name, radius) in zip(axes, [('L1', 1.5), ('L2', 1.5)]):
    ax.contour(X, Y, Z, levels=15, cmap='RdYlBu_r', alpha=0.6)
    ax.plot(2, 2, 'k*', markersize=15)
    ax.text(2.1, 2.2, 'unconstrained\nminimum', fontsize=9)

    if norm_name == 'L1':
        t = np.linspace(0, 1, 100)
        cx = np.concatenate([t, 1-t, -t, t-1]) * radius
        cy = np.concatenate([1-t, -t, t-1, t]) * radius
        ax.plot(cx, cy, 'b-', lw=2)
        ax.fill(cx, cy, alpha=0.15, color='blue')
        ax.plot(radius, 0, 'ro', markersize=10)
        ax.annotate('Constrained\nsolution (sparse!)', xy=(radius, 0),
                    xytext=(radius+0.5, 1), fontsize=10,
                    arrowprops=dict(arrowstyle='->', color='red'))
    else:
        circle = plt.Circle((0, 0), radius, fill=False, color='red', lw=2)
        ax.add_patch(circle)

    ax.set_title(f'{norm_name} Constraint')
    ax.set_aspect('equal')
    ax.set_xlim(-2, 4); ax.set_ylim(-2, 4)

plt.tight_layout()
plt.show()

---
## 9. Putting It All Together — Linear Algebra in ML

Here's how each concept appears in machine learning:

| Concept | Where It Shows Up in ML |
|---------|------------------------|
| **Vectors** | Each data point, each row of weights |
| **Dot product** | Neuron computation, cosine similarity, attention |
| **Matrix multiplication** | Layer transformations in neural networks |
| **Transpose** | Computing covariance, backpropagation |
| **Inverse** | Solving linear regression (normal equation) |
| **Determinant** | Checking if a system has a unique solution |
| **Eigenvalues/vectors** | PCA, spectral clustering, Google PageRank |
| **Norms** | Regularization (L1 = Lasso, L2 = Ridge) |

In [ ]:
np.random.seed(0)
X = np.column_stack([np.ones(50), np.random.rand(50) * 10])
true_weights = np.array([2, 3])
y = X @ true_weights + np.random.randn(50) * 2

w = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"True weights: {true_weights}")
print(f"Recovered weights (via normal equation): {np.round(w, 2)}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[:, 1], y, alpha=0.6, label='Data')
x_line = np.linspace(0, 10, 100)
ax.plot(x_line, w[0] + w[1] * x_line, 'r-', lw=2, label=f'Fit: y = {w[0]:.1f} + {w[1]:.1f}x')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Linear Regression solved with Linear Algebra\n(Normal Equation: w = (XᵀX)⁻¹Xᵀy)')
ax.legend()
plt.show()

---
## 10. NumPy Quick Reference for Linear Algebra

| Operation | NumPy |
|-----------|-------|
| Create vector | `np.array([1, 2, 3])` |
| Dot product | `np.dot(a, b)` or `a @ b` |
| Create matrix | `np.array([[1,2],[3,4]])` |
| Matrix multiply | `A @ B` |
| Transpose | `A.T` |
| Identity | `np.eye(n)` |
| Inverse | `np.linalg.inv(A)` |
| Determinant | `np.linalg.det(A)` |
| Eigenvalues | `np.linalg.eig(A)` |
| L1 norm | `np.linalg.norm(v, ord=1)` |
| L2 norm | `np.linalg.norm(v)` |